In [1]:
import json
import re
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForVision2Seq, AutoProcessor

DATA_ROOT = Path('/media/chahar/48e17169-ad03-49a2-8ad7-9f071aaf3dde')
FOODSAM_COUNTS_DIR = DATA_ROOT / 'foodsam_item_counts'
NUTRITION_VOLUME_DIR = DATA_ROOT / 'nutrition5k_volume'
OUTPUT_CSV = Path('foodqwen_macro_estimates.csv')
FAILURE_CSV = Path('foodqwen_macro_failures.csv')

model_id = 'AdaptLLM/food-Qwen2.5-VL-3B-Instruct'

processor = AutoProcessor.from_pretrained(model_id)

if torch.cuda.is_available():
    dtype = torch.float16
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    dtype = torch.float16
    device = torch.device('mps')
else:
    dtype = torch.float32
    device = torch.device('cpu')

model = AutoModelForVision2Seq.from_pretrained(model_id, torch_dtype=dtype)
model.to(device)
model.eval()

print(f"Model loaded on {device} with dtype {dtype}.")


/home/chahar/miniconda3/envs/food_cal/lib/python3.11/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded on cuda with dtype torch.float16.


In [3]:
def collect_dish_ids():
    count_ids = {p.stem for p in FOODSAM_COUNTS_DIR.glob('dish_*.json')}
    volume_ids = {p.name for p in NUTRITION_VOLUME_DIR.iterdir() if p.is_dir() and p.name.startswith('dish_')}
    dish_ids = sorted(count_ids & volume_ids)
    if not dish_ids:
        raise RuntimeError('No overlapping dish IDs found between FoodSAM counts and Nutrition5k volume outputs.')
    return dish_ids

def load_counts(dish_id):
    with open(FOODSAM_COUNTS_DIR / f'{dish_id}.json', 'r') as fp:
        return json.load(fp)

def load_category_volumes(dish_id):
    csv_path = NUTRITION_VOLUME_DIR / dish_id / 'volumes_per_category.csv'
    if not csv_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    numeric_cols = ['volume_m3', 'volume_ml', 'mean_height_cm', 'max_height_cm']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def build_prompt(dish_id, counts_data, volume_df):
    item_lines = []
    for item in counts_data.get('items', []):
        label = item.get('label', 'unknown')
        count = item.get('count', 0)
        mask_ratio_sum = item.get('mask_ratio_sum', 0.0)
        item_lines.append(f"{label}: count={count}, mask_ratio_sum={mask_ratio_sum:.4f}")
    if not item_lines:
        item_lines = ['No detected items']
    if not volume_df.empty:
        volume_lines = [
            f"{row.get('category', 'unknown')}: volume_ml={row.get('volume_ml', float('nan')):.1f}, mean_height_cm={row.get('mean_height_cm', float('nan')):.2f}"
            for _, row in volume_df.iterrows()
        ]
    else:
        volume_lines = ['No category volumes available']
    prompt_sections = [
        f'Dish ID: {dish_id}',
        f"Total food segments: {counts_data.get('total_food_segments', 'unknown')}",
        'Detected food categories with counts and mask ratios:',
        *[f"- {line}" for line in item_lines],
        'Category-level volume estimates (ml):',
        *[f"- {line}" for line in volume_lines],
        'Estimate dish-level macronutrients (carbs_g, protein_g, fat_g) and calories_kcal.',
        'Provide realistic, non-negative estimates using common food nutrition knowledge.',
        'Respond with a single JSON object only, using this schema:',
        '{"dish_id": "<dish_id>", "carbs_g": <float>, "protein_g": <float>, "fat_g": <float>, "calories_kcal": <float>}.',
        'Use grams (g) for macronutrients and kilocalories (kcal) for energy.'
    ]
    return "\n".join(prompt_sections)

def extract_macros(generated_text, fallback_dish_id):
    if not generated_text:
        return None
    match = re.search(r"\{.*\}", generated_text, re.S)
    if not match:
        return None
    try:
        payload = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    def to_float(value):
        if value is None:
            return None
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            cleaned_numbers = re.findall(r"[-+]?[0-9]*\.?[0-9]+", value)
            if cleaned_numbers:
                try:
                    return float(cleaned_numbers[0])
                except ValueError:
                    return None
        return None

    macros = {
        'dish_id': payload.get('dish_id', fallback_dish_id),
        'carbs_g': to_float(payload.get('carbs_g', payload.get('carbs'))),
        'protein_g': to_float(payload.get('protein_g', payload.get('protein'))),
        'fat_g': to_float(payload.get('fat_g', payload.get('fat'))),
        'calories_kcal': to_float(payload.get('calories_kcal', payload.get('calories'))),
    }
    numeric_keys = ['carbs_g', 'protein_g', 'fat_g', 'calories_kcal']
    if any(macros[key] is None for key in numeric_keys):
        return None
    for key in numeric_keys:
        macros[key] = max(0.0, macros[key])
    return macros

def infer_macros(dish_id, counts_data, volume_df, max_new_tokens=256):
    prompt = build_prompt(dish_id, counts_data, volume_df)
    messages = [
        {'role': 'system', 'content': 'You are a nutrition scientist. Base your estimates on the provided counts and category volumes. Answer with JSON only.'},
        {'role': 'user', 'content': [{'type': 'text', 'text': prompt}]},
    ]
    chat_prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[chat_prompt], return_tensors='pt', padding=True)
    inputs = {
        key: (value.to(device) if isinstance(value, torch.Tensor) else value)
        for key, value in inputs.items()
    }
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    prompt_length = inputs['input_ids'].shape[-1]
    response_ids = generated_ids[:, prompt_length:]
    if response_ids.shape[1] == 0:
        generated_text = ''
    else:
        tokenizer = getattr(processor, 'tokenizer', None) or processor
        generated_text = tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0].strip()
    macros = extract_macros(generated_text, dish_id)
    return macros, generated_text


dish_ids = collect_dish_ids()
print(f'Found {len(dish_ids)} dishes with both counts and volume data.')

MAX_DISHES = None  # set to an integer to limit the run for testing
if MAX_DISHES is not None:
    dish_ids = dish_ids[:MAX_DISHES]
    print(f'Processing first {len(dish_ids)} dishes due to MAX_DISHES setting.')

results = []
failures = []

for dish_id in tqdm(dish_ids, desc='Estimating macros'):
    counts_data = load_counts(dish_id)
    volume_df = load_category_volumes(dish_id)
    macros, raw_text = infer_macros(dish_id, counts_data, volume_df)
    if macros:
        results.append(macros)
    else:
        failures.append({'dish_id': dish_id, 'raw_response': raw_text})

results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df[['dish_id', 'carbs_g', 'protein_g', 'fat_g', 'calories_kcal']]
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f'Saved {len(results_df)} dish macro estimates to {OUTPUT_CSV.resolve()}.')
else:
    print('No macro estimates were produced.')

if failures:
    failure_df = pd.DataFrame(failures)
    failure_df.to_csv(FAILURE_CSV, index=False)
    print(f"{len(failures)} dishes failed to parse. Raw LLM responses saved to {FAILURE_CSV.resolve()}.")
else:
    print('All dishes parsed successfully.')

Found 3484 dishes with both counts and volume data.


Estimating macros:   0%|          | 0/3484 [00:00<?, ?it/s]

Saved 3484 dish macro estimates to /home/chahar/food_new/llms_approach/foodqwen_macro_estimates.csv.
All dishes parsed successfully.
